# 02 · A network that computes: a direction detector

In which direction does a dot move across four pixels? A layer of two `lif` neurons answers it, using what notebook 01 showed: spikes that arrive together fire a neuron, and delays make them arrive together. Runs on Brian2; cluster participants also run it on the chip (section 4).

**Your task** marks the exercise, **Discuss** questions we take up together, ▶ optional background.

## 0. Setup

In [ ]:
# --- libraries ---
import numpy as np  # arrays and numbers
import matplotlib.pyplot as plt  # figures
from matplotlib.animation import FuncAnimation, PillowWriter  # animations, saved as GIFs
from IPython.display import Image, display  # shows the GIFs in the notebook
from spinnaker2 import brian2_sim, snn  # py-spinnaker2: the Brian2 backend and the network objects

# --- backend: where the networks run ---
brian2 = brian2_sim.Brian2Backend()  # the simulator on this computer; section 4 runs the network on the chip

# --- figure style ---
plt.rcParams.update({"axes.grid": True, "axes.axisbelow": True, "grid.color": "0.88", "grid.linewidth": 0.6})  # light grey grid in all figures

### Helpers for figures and animations

Functions that only draw; nothing about the network.

In [ ]:
def animate_input(inputs, n_pixels, filename, steps=15, trail=True):
    """Save and show a GIF of inputs {title: {pixel: [time steps]}}: the pixels in a row, each lighting up when it fires."""
    fig, axes = plt.subplots(1, len(inputs), figsize=(10, 2))
    images, traces = [], []
    for ax, title in zip(axes, inputs):
        images.append(ax.imshow(np.zeros((1, n_pixels)), cmap="gray_r", vmin=0, vmax=1))
        ax.set(title=title, xticks=range(n_pixels), yticks=[], xlabel="pixel")
        ax.grid(False)
        traces.append(np.zeros(n_pixels))
    step_text = fig.suptitle("")

    def update(t):
        for i, spikes in enumerate(inputs.values()):
            fired = np.array([1.0 if t in spikes.get(p, []) else 0.0 for p in range(n_pixels)])
            traces[i] = (0.6 if trail else 0.0) * traces[i] + fired  # older events fade (only for the eye)
            images[i].set_data(np.clip(traces[i], 0, 1)[None, :])
        step_text.set_text(f"time step {t}")
        return images

    FuncAnimation(fig, update, frames=steps, interval=400).save(filename, writer=PillowWriter(fps=2.5))
    plt.close(fig)
    display(Image(filename=filename))


def plot_raster(inputs, n_pixels, xlim=15):
    """Raster plots of inputs {title: {pixel: [time steps]}}, side by side: one row per pixel, one mark per spike."""
    fig, axes = plt.subplots(1, len(inputs), figsize=(10, 3), sharey=True, squeeze=False)
    for ax, (title, spikes) in zip(axes[0], inputs.items()):
        ax.eventplot([spikes.get(p, []) for p in range(n_pixels)], lineoffsets=range(n_pixels), linelengths=0.6, colors="black")
        ax.set(title=title, xlabel="time step (1 step = 1 ms)", xlim=(0, xlim), yticks=range(n_pixels))
    axes[0][0].set_ylabel("pixel")
    plt.tight_layout()
    plt.show()


def plot_potentials(results, threshold, names):
    """Plot the detectors' potentials for each run in results {title: (spikes, v)}, one panel per run."""
    fig, axes = plt.subplots(1, len(results), figsize=(10, 4), sharey=True, squeeze=False)
    for ax, (title, (spikes, v)) in zip(axes[0], results.items()):
        for d, name in enumerate(names):
            color = f"C{d}"  # the same colour for a detector in every panel
            ax.plot(v[d], ".-", color=color, label=name)
            for t in spikes.get(d, []):  # v is recorded after the reset; show the value before it as an open circle
                before = v[d][t] + threshold  # reset_by_subtraction: before = after + threshold
                ax.plot([t - 1, t], [v[d][t - 1], before], ":", color=color)
                ax.plot(t, before, "o", mfc="white", mec=color, ms=8)
                ax.plot(t, before + 0.8, "v", color=color, ms=10)  # the spike
        ax.axhline(threshold, ls="--", c="0.5")
        ax.set(title=title, xlabel="time step (1 step = 1 ms)")
    axes[0][0].set_ylabel(f"membrane potential v (threshold = {threshold:g})")
    axes[0][0].legend(loc="upper left")
    plt.tight_layout()
    plt.show()


def animate_detector(input_spikes, spikes, v, n_pixels, threshold, names, title, filename, trail=True):
    """Save and show a GIF of one run: pixels firing (left) and the detectors' potentials building up (right)."""
    steps = len(v[0])
    fired = np.zeros((steps, n_pixels))
    for pixel, times in input_spikes.items():
        for t in times:
            fired[t, pixel] = 1

    fig, (ax_px, ax_v) = plt.subplots(1, 2, figsize=(10, 4), gridspec_kw={"width_ratios": [1, 2]})
    image = ax_px.imshow(np.zeros((1, n_pixels)), cmap="gray_r", vmin=0, vmax=1)
    ax_px.set(xticks=range(n_pixels), yticks=[], xlabel="pixel", title=title)
    ax_px.grid(False)
    lines = [ax_v.plot([], [], ".-", color=f"C{d}", label=name)[0] for d, name in enumerate(names)]
    ax_v.axhline(threshold, ls="--", c="0.5")
    ax_v.set(xlim=(0, steps - 1), ylim=(0, threshold + 3), xlabel="time step (1 step = 1 ms)", ylabel="membrane potential v")
    ax_v.legend(loc="upper left")
    markers = [ax_v.plot([], [], "v", color=f"C{d}", ms=12)[0] for d in range(len(names))]  # one marker set per detector, in its colour
    trace = np.zeros(n_pixels)

    def update(t):
        nonlocal trace
        trace = (0.6 if trail else 0.0) * trace + fired[t]  # older events fade (only for the eye)
        image.set_data(np.clip(trace, 0, 1)[None, :])
        for d, line in enumerate(lines):
            line.set_data(range(t + 1), v[d][:t + 1])  # v exactly as recorded (after a possible reset)
        for d, marker in enumerate(markers):  # spike markers, only a visual help
            done = [s for s in spikes.get(d, []) if s <= t]
            marker.set_data(done, [threshold + 1.5] * len(done))
        ax_v.set_title(f"time step {t}")
        return lines + markers + [image]

    FuncAnimation(fig, update, frames=steps, interval=400).save(filename, writer=PillowWriter(fps=2.5))
    plt.close(fig)
    display(Image(filename=filename))

## 1. The input: a dot moving across four pixels

Four pixels in a row, like a tiny strip of an event camera. When a dot passes a pixel, the pixel fires one spike. A dot moving to the right makes pixel 0 fire first, then pixels 1, 2 and 3; a dot moving to the left reverses the order. The function `moving_dot` makes this input for a direction and a speed: `step` is the number of time steps the dot needs from one pixel to the next.

In [ ]:
# --- the input ---
N_PIXELS = 4


def moving_dot(direction, step=1, start=5):
    """Spike times of the pixels, {pixel: [time step]}, for a dot moving "right" or "left".

    Example: moving_dot("right") -> {0: [5], 1: [6], 2: [7], 3: [8]}
    """
    order = range(N_PIXELS) if direction == "right" else reversed(range(N_PIXELS))  # the order in which the dot passes the pixels
    return {pixel: [start + k * step] for k, pixel in enumerate(order)}  # the k-th pixel fires k * step steps after start


# --- show it ---
show_trail = True  # animations: True: older events fade out (for the eye only); False: only the events of each step, as the network gets them
animate_input({"dot moving right": moving_dot("right"), "dot moving left": moving_dot("left")}, N_PIXELS, "detector_input.gif", trail=show_trail)

The same input as a raster, one row per pixel. Every pixel fires exactly once in both directions; only the order differs.

In [ ]:
# --- the spike times, printed and as rasters ---
print("dot moving right:", moving_dot("right"))
print("dot moving left: ", moving_dot("left"))
plot_raster({"dot moving right": moving_dot("right"), "dot moving left": moving_dot("left")}, N_PIXELS)

## 2. The detector layer: weights and delays as matrices

**Your task:** wire up two `lif` neurons so that one fires only for a dot moving right and the other only for a dot moving left. Each neuron is a detector, connected to every pixel; you choose the delays of these connections.

- **`W` (weights) and `D` (delays) are matrices**, as for a layer in a deep network: one row per detector (0: rightward, 1: leftward), one column per pixel (0 to 3). The weights are given; `D` is yours to fill in.
- **A row is not a connection:** a row of `D` holds the four delays of one detector, one per pixel. The cell builds the connection list `[pre, post, weight, delay]` from `W` and `D`.
- **Ranges:** weights −15 to 15, delays 0 to 7 time steps.

**Done when:** in section 3, the dot moving right makes only the rightward detector fire, and the dot moving left only the leftward one.

In [ ]:
# --- fixed: names and neuron parameters of the two detectors ---
DETECTOR_NAMES = ["rightward", "leftward"]  # row 0 and row 1 of W and D
detector_params = {"threshold": 10.0, "alpha_decay": 0.8, "reset": "reset_by_subtraction"}  # keep 80 % of v per step; after a spike, subtract the threshold

# ============================== YOUR TASK ==============================
# W: weights, D: delays in time steps (0 to 7). Matrices, not connections:
# one row per detector, one column per pixel.
#       pixel: 0  1  2  3
W = np.array([[3, 3, 3, 3],   # row 0, rightward detector: weights (given)
              [3, 3, 3, 3]])  # row 1, leftward detector: weights (given)
D = np.array([[0, 0, 0, 0],   # row 0, rightward detector: delays    <-- FILL IN
              [0, 0, 0, 0]])  # row 1, leftward detector: delays     <-- FILL IN
# =======================================================================

# --- build the connection list [pre, post, weight, delay] from W and D ---
connections = []
for post in range(W.shape[0]):  # every detector (row) ...
    for pre in range(W.shape[1]):  # ... gets a connection from every pixel (column)
        connections.append([pre, post, int(W[post, pre]), int(D[post, pre])])

print(len(connections), "connections [pre, post, weight, delay]:")
width = max(len(name) for name in DETECTOR_NAMES)  # pad the names so the lists line up
for post, name in enumerate(DETECTOR_NAMES):
    print(f"  {name + ':':{width + 1}}", *[c for c in connections if c[1] == post])  # c[1] is post: the detector

<details><summary>Solution: the delays of both detectors</summary>

- Rightward: pixel 0 fires first and pixel 3 last, so pixel 0 waits longest. Row 0 of `D` is `[3, 2, 1, 0]`.
- Leftward: pixel 3 fires first and pixel 0 last, so pixel 3 waits longest. Row 1 of `D` is `[0, 1, 2, 3]`.

With a dot moving one pixel per step, all four spikes then arrive in the same step.

</details>

## 3. Run it and read the answer

`show` runs the network for each input and prints which detector fired: the network's answer. After changing anything in section 2, run its cell again.

<details><summary>Reference: py-spinnaker2 in detail (what <code>run_detector</code> uses, all options, where to learn more)</summary>

`run_detector` below builds the network from the four building blocks of py-spinnaker2. Notebook 01 used each of them once; here they are with their options, as of py-spinnaker2 0.8.5, and where the vendor explains them in depth.

**The four building blocks in one example**

```python
pixels = snn.Population(size=4, neuron_model="spike_list", params={0: [5], 1: [6]}, name="pixels")  # 1. populations
detectors = snn.Population(size=2, neuron_model="lif", params={"threshold": 10.0}, name="detectors", record=["spikes", "v"])
proj = snn.Projection(pre=pixels, post=detectors, connections=[[0, 0, 3, 3], [1, 0, 3, 2]])  # 2. projections
net = snn.Network("example")  # 3. the network
net.add(pixels, detectors, proj)
brian2.run(net, 25)  # 4. a backend runs it
spikes = detectors.get_spikes()  # the results are stored in the population
```

**`snn.Population(size, neuron_model, params, name=None, record=[])`**: a group of neurons of one model.

- `size`: the number of neurons, addressed by index 0 to `size` − 1.
- `neuron_model`: the model, by name. `"lif"` in this workshop, `"spike_list"` for inputs; the others (`lif_no_delay`, `alif`, `lif_prob`, `izhikevich`, …) are listed in the [API docs, neuron models](https://spinnaker2.gitlab.io/py-spinnaker2/api_reference/neuron-models.html).
- `params` of `spike_list`: `{neuron index: [time steps]}`; unlisted neurons stay silent.
- `params` of `lif`: `threshold` (default 1.0), `alpha_decay` (0.8), `i_offset` (0), `v_init` (0), `v_reset` (0) and `reset` (`"reset_by_subtraction"`, the default, or `"reset_to_v_reset"`). A number applies to every neuron, a list sets one value per neuron. A misspelled name stops with an error.
- `name`: must be unique within a network.
- `record`: what to keep: `"spikes"`, `"v"` and, on the chip only, `"time_done"` (section 4). Only what is recorded can be read back.
- **Recording costs memory on the chip.** Recorded values stay in the memory of the processing element (128 KB per PE, shared with the parameters and synapses) until the run ends: `v` takes 4 bytes per neuron and time step, spikes about one bit per neuron and step. So a network that runs on Brian2 can fail on the chip before it starts, with a `MemoryError` ("PE memory exceeded"): 100 `lif` neurons recording `v` over 1,000 steps need about 480 KiB. Remedies: record only what you need (often just `"spikes"`), only the final potential (`"v_last"`), or store the recordings in the chip's DRAM (`record_on_dram=True`, an argument of `Population`). `run(..., mapping_only=True)` checks whether a network fits without running it. More in the workshop notes: [time and memory limits](../01_knowledge_base/02_hardware_energy.md#7-limits-in-practice-when-a-network-doesnt-fit).

**`snn.Projection(pre, post, connections, name=None)`**: the synapses from one population to another.

- `connections`: a list with one `[pre index, post index, weight, delay]` per synapse.
- For `lif`: weights are whole numbers from −15 to 15 (negative ones inhibit), delays 0 to 7 time steps. Values outside the range are clipped and fractions cut off, without a warning.
- A spike sent in step $t$ arrives in step $t + 1 + \text{delay}$.
- Several projections into the same population add up.

**`snn.Network(name)`**: everything that runs together. `net.add(...)` takes populations and projections, each once, in any order; every population a projection connects must be added too, otherwise `run` fails. `net.reset()` clears all results; without it, a new run merges its results into the old ones.

**Running: `backend.run(net, time_steps)`** returns nothing; the results are stored in the populations.

- `brian2_sim.Brian2Backend()`: a simulator on this computer, in seconds, with the chip's integer weights and delays.
- `hardware.SpiNNcloud48NodeBoard()`: one 48-chip board (section 4). Useful options of `run`: `sys_tick_in_s`, the real-time length of a step (default 0.001 s; longer when a core cannot finish its work within 1 ms), and `mapping_only=True`, which only works out how the network would be placed on the cores, without running it.

**Reading the results**

- `pop.get_spikes()`: `{neuron index: spike times}`. A neuron without spikes may be missing, so use `.get(i, [])`.
- `pop.get_voltages()`: `{neuron index: v in every step}`, recorded after a possible reset.
- `pop.get_time_done_times()`: chip only, `{processing element: share of each step spent computing}` (section 4).

**Where to learn the syntax in depth** (vendor material for py-spinnaker2 0.8.5, SpiNNcloud, Apache-2.0)

- [Tutorial 00](https://gitlab.com/spinnaker2/py-spinnaker2/-/blob/v0.8.5/tutorials/deep_dive/00_welcome.ipynb), sections 4 and 5: the four building blocks, and a first network step by step.
- [Tutorial 01](https://gitlab.com/spinnaker2/py-spinnaker2/-/blob/v0.8.5/tutorials/deep_dive/01_core_concepts.ipynb): populations, neuron models, shared and per-neuron parameters, spike sources.
- [Tutorial 02](https://gitlab.com/spinnaker2/py-spinnaker2/-/blob/v0.8.5/tutorials/deep_dive/02_connectivity.ipynb): projections, weights and delays precisely, connection patterns, from a weight matrix to a connection list, combining projections.
- [Tutorial 04](https://gitlab.com/spinnaker2/py-spinnaker2/-/blob/v0.8.5/tutorials/deep_dive/04_hardware_allocation.ipynb): the board classes and how a board is chosen.
- [Tutorial 05](https://gitlab.com/spinnaker2/py-spinnaker2/-/blob/v0.8.5/tutorials/deep_dive/05_debugging.ipynb): what `run` does on the chip, `mapping_only`, logging, profiling, common issues.
- User guide: [SNN introduction](https://spinnaker2.gitlab.io/py-spinnaker2/user_guide/snn-intro.html) (building, running, reading results, `time_done`) and [advanced SNN usage](https://spinnaker2.gitlab.io/py-spinnaker2/user_guide/snn-advanced.html) (timing and `sys_tick_in_s`).
- API reference: [snn](https://spinnaker2.gitlab.io/py-spinnaker2/api_reference/snn-api.html), [neuron models](https://spinnaker2.gitlab.io/py-spinnaker2/api_reference/neuron-models.html), [hardware](https://spinnaker2.gitlab.io/py-spinnaker2/api_reference/hardware-api.html).

</details>

In [ ]:
# --- the network ---
def run_detector(input_spikes, backend=brian2, steps=25, record=("spikes", "v")):
    """Build the network for one input and run it; return the detectors' spikes, membrane potentials and population."""
    pixels = snn.Population(size=N_PIXELS, neuron_model="spike_list", params=input_spikes, name="pixels")  # sends the input spikes
    detectors = snn.Population(size=W.shape[0], neuron_model="lif", params=detector_params, name="detectors", record=list(record))  # one lif neuron per row of W
    net = snn.Network("direction_detector")
    net.add(pixels, detectors, snn.Projection(pre=pixels, post=detectors, connections=connections))  # pixels -> detectors, with W and D of section 2
    backend.run(net, steps)
    return detectors.get_spikes(), detectors.get_voltages(), detectors


# --- run it for several inputs and read the answer ---
def show(inputs):
    """Run the detector for each input {title: input_spikes}, print which detectors fired and plot their potentials."""
    results = {}
    for title, input_spikes in inputs.items():
        spikes, v, _ = run_detector(input_spikes)
        fired = [DETECTOR_NAMES[d] for d in range(W.shape[0]) if len(spikes.get(d, [])) > 0]  # a detector answers if it fired at least once
        print(f"{title}: {', '.join(fired) if fired else 'no detector'} fired")  # the network's answer
        results[title] = (spikes, v)
    plot_potentials(results, detector_params["threshold"], DETECTOR_NAMES)


# --- the two directions ---
show({"dot moving right": moving_dot("right"), "dot moving left": moving_dot("left")})

Triangles mark spikes. The potential is recorded after the reset, so in the spike step the line drops instead of crossing the threshold; in the plot above, the open circle shows the value just before the reset, computed from the recorded one (recorded + threshold). The animation shows the same run step by step, with `v` exactly as recorded. The fading of the pixels is only for the eye, since the network's memory is the potential of the detectors (`show_trail = False` in section 1 switches it off).

In [ ]:
# --- run one input and animate it ---
input_spikes = moving_dot("right", step=1)
spikes, v, _ = run_detector(input_spikes, steps=25)  # 25 steps, as in section 3
animate_detector(input_spikes, spikes, v, N_PIXELS, detector_params["threshold"], DETECTOR_NAMES,
                 "dot moving right, 1 pixel per step", "detector_right.gif", trail=show_trail)

**Discuss:**

- With the dot moving right or left, every pixel fires exactly once, in both directions. Where is the information that tells the two apart?
- The detector layer is a weight matrix, like a layer in a deep network. Could a layer of a deep network that sees only which pixels were active tell the two directions apart? What does the spiking layer have that it lacks?

*Your answers:*

- …
- …

## 4. Run it on the SpiNNaker2 chip (cluster participants)

**The HPI setup:** one frame with 6 boards of 48 SpiNNaker2 chips each, attached to one node of the HPI cluster. py-spinnaker2 calls such a board `hardware.SpiNNcloud48NodeBoard()` (`hardware.SpiNNaker2Chip()` is for boards with a single chip, which HPI does not have). A run reserves a whole board, uses only the cores the network needs and takes about a minute, mostly for powering up and booting; with 6 boards, at most 6 runs at a time, and a run waits if all are busy. That is why everything else runs on Brian2.

Set `USE_HARDWARE = True` and run the cell once per pair, now. Without cluster access, read along.

What the cell prints, from top to bottom:

- `Number of cores needed: 2` of the `7104` available on the board: one processing element (PE) for the pixels, one for the detectors.
- `Sim duration`: how long the board ran the network; the minute before it goes to powering up, booting and loading.
- The detectors' spikes and whether the answer matches Brian2, then both runs plotted side by side.
- `busy … %`: how much of each time step the detectors' PE was working (`time_done`, explained below the cell). Brian2 cannot show this: it has no real time.

In [ ]:
# --- backend ---
USE_HARDWARE = False  # True: a SpiNNaker2 board (HPI cluster only); False: Brian2, as a dry run

if USE_HARDWARE:
    from spinnaker2 import hardware  # the chip backend, only needed on the cluster

    hw = hardware.SpiNNcloud48NodeBoard()  # one 48-chip board, reserved for this run
else:
    hw = brian2_sim.Brian2Backend()
print("Running on backend:", type(hw).__name__)

# --- build and run: the detector for a dot moving right ---
input_spikes = moving_dot("right")
record = ["spikes", "v", "time_done"] if USE_HARDWARE else ["spikes", "v"]  # time_done only exists on the chip
spikes, v, detectors = run_detector(input_spikes, backend=hw, record=record)

# --- compare with Brian2: printed and plotted with the same function as in section 3 ---
ref_spikes, ref_v, _ = run_detector(input_spikes)  # the same input on Brian2
result = {d: list(ts) for d, ts in spikes.items() if len(ts)}  # only detectors that fired
reference = {d: list(ts) for d, ts in ref_spikes.items() if len(ts)}
print("detector spikes:", {DETECTOR_NAMES[d]: ts for d, ts in result.items()})
print("same answer as on Brian2" if result == reference else "different from Brian2")
title = "recorded on the SpiNNaker2 chip" if USE_HARDWARE else "Brian2 (dry run)"
plot_potentials({title: (spikes, v), "recorded on Brian2": (ref_spikes, ref_v)}, detector_params["threshold"], DETECTOR_NAMES)

# --- how busy was the detectors' processing element (PE)? chip only ---
if USE_HARDWARE:
    for pe, fraction in detectors.get_time_done_times().items():  # {PE name: share of each step spent working, one value per step}
        print(f"{pe}: busy {100 * fraction.mean():.1f} % of each 1 ms step on average (about {1000 * fraction.mean():.0f} µs), "
              f"at most {100 * fraction.max():.1f} %")

**How to read the last line** (`time_done`, chip only):

- `Chip_0_0_QPE_1_1_1` names the processing element (PE) that ran the detectors: chip (0, 0) on the board, quad (1, 1) on that chip, PE 1 in the quad. A quad is a group of 4 PEs; a chip has 38 of them, 152 PEs in total.
- A timer wakes each PE once per time step, every 1 ms. The PE processes the spikes that arrived, updates its neurons, sends their spikes, and sleeps until the next tick. `time_done` records, for every step, the share of the 1 ms the PE was working.
- `busy 0.3 %` means about 3 µs of work per step, for 2 neurons and a handful of spikes. The rest of the millisecond the PE sleeps: the chip runs in real time, one step per millisecond, however little there is to do.
- The work grows with the number of neurons on the PE (up to 256 for `lif`) and with the spikes that arrive. If a step ever needed more than 1 ms, the PE would fall behind real time: the run does not stop, but logs "Deadline missed". `run(..., sys_tick_in_s=...)` then makes every step longer.
- What a mostly sleeping PE means for the energy of a run follows in the energy demo at the end of the afternoon.

---

**Summary:** a layer of two `lif` neurons, written as a weight and a delay matrix, tells the direction of a moving dot from the timing of four spikes: each detector is a coincidence detector, and its delays decide which direction and speed it waits for. On the chip, it gives the same answer while its core sleeps most of every time step. After the break, a network learns this task itself: it is trained in PyTorch, first on the same four pixels (notebook 03), then on a larger 8 × 8 sensor with noise (notebook 04), and the NIR import builds the populations and projections from its weight matrices.

**Go deeper:** delays and delay lines: tutorial 02, section 2; building connection lists from matrices: tutorial 02, section 4; recording what the chip did: `time_done` in the [user guide, execution times](https://spinnaker2.gitlab.io/py-spinnaker2/user_guide/snn-intro.html#execution-times-time_done), profiling in tutorial 05 and the [user guide, profiling](https://spinnaker2.gitlab.io/py-spinnaker2/user_guide/profiling.html). Workshop notes: [synapses](../01_knowledge_base/01_primer_snn_basics.md#4-synapses), [what a PE does per time step](../01_knowledge_base/02_hardware_energy.md#2-the-processing-element), [time and memory limits](../01_knowledge_base/02_hardware_energy.md#7-limits-in-practice-when-a-network-doesnt-fit).

---

**End of notebook 02.** Then the break; after it, from PyTorch to the chip.